In [1]:
import requests
import json
import time
import csv
import re
from typing import List, Dict

SUBREDDITS = [
    "developersIndia",
    "cscareerquestions",
    "cscareerquestionsIND",
    "programming",
    "learnprogramming",
    "worklifebalance",
    "careerguidance",
    "ExperiencedDevs",
    "ITCareerQuestions",
    "mentalhealth",
]


HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0"
}

MAX_POSTS = 500
REQUEST_DELAY = 1.5



def scrape_posts() -> List[Dict]:
    collected = []

    for subreddit in SUBREDDITS:
        if len(collected) >= MAX_POSTS:
            break

        print(f"Fetching from r/{subreddit}")
        url = f"https://www.reddit.com/r/{subreddit}/new.json?limit=100"

        try:
            res = requests.get(url, headers=HEADERS, timeout=15)
            res.raise_for_status()
            data = res.json()

            for post in data["data"]["children"]:
                if len(collected) >= MAX_POSTS:
                    break

                post_data = post["data"]
                title = post_data.get("title", "")
                body = post_data.get("selftext", "")

                full_text = (title + " " + body).strip()

                pass

                comments_url = f"https://www.reddit.com{post_data['permalink']}.json?limit=10"
                comments_res = requests.get(comments_url, headers=HEADERS, timeout=15)
                comments_json = comments_res.json()

                comments = []
                for c in comments_json[1]["data"]["children"]:
                    if c["kind"] == "t1":
                        text = c["data"].get("body", "")
                        if text:
                            comments.append(text)

                collected.append({
                    "post_text": full_text,
                    "comments": comments,
                })

                time.sleep(REQUEST_DELAY)

        except Exception as e:
            print(f"Error in r/{subreddit}: {e}")

    return collected



def save_dataset(data: List[Dict], filename="toy_dev_stress_dataset3.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(data)} posts to {filename}")
    
def save_csv_for_ml(data, filename="dev_stress_docs.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["doc_id", "text"])

        for i, item in enumerate(data):
            combined_text = item["post_text"]
            if item["comments"]:
                combined_text += " || " + " || ".join(item["comments"])

            writer.writerow([i, combined_text])
    
    print(f"Saved CSV with {len(data)} documents")


if __name__ == "__main__":
    dataset = scrape_posts()
    save_dataset(dataset)
    save_csv_for_ml(dataset)


Fetching from r/developersIndia
Fetching from r/cscareerquestions
Error in r/cscareerquestions: Expecting value: line 1 column 1 (char 0)
Fetching from r/cscareerquestionsIND
Fetching from r/programming
Error in r/programming: Expecting value: line 1 column 1 (char 0)
Fetching from r/learnprogramming
Error in r/learnprogramming: Expecting value: line 1 column 1 (char 0)
Fetching from r/worklifebalance
Error in r/worklifebalance: 429 Client Error: Too Many Requests for url: https://www.reddit.com/r/worklifebalance/new.json?limit=100
Fetching from r/careerguidance
Error in r/careerguidance: Expecting value: line 1 column 1 (char 0)
Fetching from r/ExperiencedDevs
Error in r/ExperiencedDevs: 429 Client Error: Too Many Requests for url: https://www.reddit.com/r/ExperiencedDevs/new.json?limit=100
Fetching from r/ITCareerQuestions
Error in r/ITCareerQuestions: 429 Client Error: Too Many Requests for url: https://www.reddit.com/r/ITCareerQuestions/new.json?limit=100
Fetching from r/mentalheal